## Update Video Lines

Reads every video service on a Video Server and keeps one line per service in the **Video Lines** layer: the sensor's path, the service name and the service URL. The drone video player lists the lines in view and plays the service behind the one you pick.

Run it after publishing new video services, or put it on a schedule.

In [ ]:
# Settings for this deployment. Change them here, nowhere else.
VIDEO_SERVER_URL = "{{video.server_url}}"  # the Video Server's services directory, e.g. https://host/video/rest/services
LINE_ITEM_ID = "{{video_lines.itemId}}"  # the Video Lines layer this deployment created
CHUNK = 10  # features per edit request; flight lines are long


In [ ]:
import requests
from arcgis.gis import GIS
from arcgis.features import Feature
from arcgis.geometry import Polyline

if not VIDEO_SERVER_URL:
    raise ValueError("Set VIDEO_SERVER_URL in the first cell to your Video Server's services directory.")
ROOT = VIDEO_SERVER_URL.rstrip("/")
gis = GIS("home")
line_layer = gis.content.get(LINE_ITEM_ID).layers[0]


def video_services():
    """Every video service in the directory, following its paging."""
    found, start = [], None
    while True:
        params = {"f": "json"} if start is None else {"f": "json", "q": "", "start": start, "num": 100}
        page = requests.get(ROOT, params=params, timeout=60).json()
        found.extend(page.get("results") or [])
        start = page.get("nextStart", -1)
        if start is None or start < 0:
            break
    return [s for s in found if s.get("serviceType") == "Video Service"]


def flight_line(service_name):
    """The sensor's path for one video service, or None when it has no coverage."""
    coverage = requests.get(f"{ROOT}/{service_name}/VideoServer/0/coverage", params={"f": "json"}, timeout=120).json()
    points = (coverage.get("sensorPoints") or {}).get("points") or []
    path = [[p[0], p[1]] for p in points if not (p[0] == 0 and p[1] == 0)]
    if len(path) < 2:
        return None
    return Polyline({"paths": [path], "spatialReference": {"wkid": 4326}})


wanted = {}
for service in video_services():
    name = service["serviceName"]
    line = flight_line(name)
    if line is None:
        print("no sensor path:", name)
        continue
    wanted[name] = Feature(geometry=line, attributes={"URL": f"{ROOT}/{name}/VideoServer/0", "service_name": name})
print(len(wanted), "video services with a flight line")


In [ ]:
# Update in place: the layer is never emptied, so the player always has a list.
existing = {f.attributes["service_name"]: f.attributes[line_layer.properties.objectIdField]
            for f in line_layer.query(where="1=1", out_fields="service_name", return_geometry=False).features}
adds, updates = [], []
for name, feature in wanted.items():
    if name in existing:
        feature.attributes[line_layer.properties.objectIdField] = existing[name]
        updates.append(feature)
    else:
        adds.append(feature)
deletes = [oid for name, oid in existing.items() if name not in wanted]

for label, rows in (("added", adds), ("updated", updates)):
    for i in range(0, len(rows), CHUNK):
        part = rows[i:i + CHUNK]
        result = line_layer.edit_features(adds=part) if label == "added" else line_layer.edit_features(updates=part)
        failed = [r for r in result.get("addResults", []) + result.get("updateResults", []) if not r.get("success")]
        if failed:
            print(label, "failures:", failed)
if deletes:
    line_layer.edit_features(deletes=deletes)
print(f"added {len(adds)}, updated {len(updates)}, removed {len(deletes)}")
